In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec

In [3]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [4]:
df = pd.read_csv("IMDB Dataset.csv")
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
print(df.head())
print(df.info())
print(df.columns)


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None
Index(['review', 'sentiment'], dtype='object')


In [6]:
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'<.*?>', '', text)  # remove HTML tags
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation

    words = text.split()


    return " ".join(words)

In [9]:
df['clean_review'] = df['review'].apply(clean_text)

In [10]:
df[['review', 'clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love in the time of money is a ...


In [11]:

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(df.head())

                                              review  sentiment  \
0  One of the other reviewers has mentioned that ...          1   
1  A wonderful little production. <br /><br />The...          1   
2  I thought this was a wonderful way to spend ti...          1   
3  Basically there's a family where a little boy ...          0   
4  Petter Mattei's "Love in the Time of Money" is...          1   

                                        clean_review  
0  one of the other reviewers has mentioned that ...  
1  a wonderful little production the filming tech...  
2  i thought this was a wonderful way to spend ti...  
3  basically theres a family where a little boy j...  
4  petter matteis love in the time of money is a ...  


In [12]:
for i in range(3):
    print(f"\nSentence {i}:")
    print(df['clean_review'].iloc[i].split())




Sentence 0:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'methe', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'go', 'trust', 'me', 'this', 'is', 'not', 'a', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', 'this', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', 'sex', 'or', 'violence', 'its', 'is', 'hardcore', 'in', 'the', 'classic', 'use', 'of', 'the', 'wordit', 'is', 'called', 'oz', 'as', 'that', 'is', 'the', 'nickname', 'given', 'to', 'the', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'it', 'focuses', 'mainly', 'on', 'emerald', 'city', 'an', 'experimental', 'section', 'of', 'the', 'prison', 'where', 'all', 'the', 'cell

In [13]:

X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [14]:
import gensim.downloader as api


In [15]:


word2vec_model = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [16]:
def sentence_vector(sentence):
    words = sentence.split()
    vectors = [word2vec_model[word] for word in words if word in word2vec_model]
    if len(vectors) == 0:
        return np.zeros(300)
    return np.mean(vectors, axis=0)

In [17]:
X_train_vectors = np.array([sentence_vector(text) for text in X_train])
X_test_vectors  = np.array([sentence_vector(text) for text in X_test])

X_train_tensor = torch.tensor(X_train_vectors, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_vectors, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [18]:
class SentimentClassifier(nn.Module):
    def __init__(self):
        super(SentimentClassifier, self).__init__()
        self.fc1 = nn.Linear(300, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

model = SentimentClassifier()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

In [24]:
epochs = 200
for epoch in range(epochs):
    model.train()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [1/200], Loss: 0.5692
Epoch [2/200], Loss: 0.5682
Epoch [3/200], Loss: 0.5673
Epoch [4/200], Loss: 0.5664
Epoch [5/200], Loss: 0.5654
Epoch [6/200], Loss: 0.5645
Epoch [7/200], Loss: 0.5636
Epoch [8/200], Loss: 0.5627
Epoch [9/200], Loss: 0.5617
Epoch [10/200], Loss: 0.5608
Epoch [11/200], Loss: 0.5599
Epoch [12/200], Loss: 0.5590
Epoch [13/200], Loss: 0.5581
Epoch [14/200], Loss: 0.5571
Epoch [15/200], Loss: 0.5562
Epoch [16/200], Loss: 0.5553
Epoch [17/200], Loss: 0.5544
Epoch [18/200], Loss: 0.5535
Epoch [19/200], Loss: 0.5526
Epoch [20/200], Loss: 0.5517
Epoch [21/200], Loss: 0.5508
Epoch [22/200], Loss: 0.5499
Epoch [23/200], Loss: 0.5490
Epoch [24/200], Loss: 0.5481
Epoch [25/200], Loss: 0.5472
Epoch [26/200], Loss: 0.5463
Epoch [27/200], Loss: 0.5455
Epoch [28/200], Loss: 0.5446
Epoch [29/200], Loss: 0.5437
Epoch [30/200], Loss: 0.5428
Epoch [31/200], Loss: 0.5419
Epoch [32/200], Loss: 0.5410
Epoch [33/200], Loss: 0.5402
Epoch [34/200], Loss: 0.5393
Epoch [35/200], Loss: 0

In [25]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
predicted = (predictions >= 0.5).float()
accuracy = accuracy_score(y_test_tensor.numpy(), predicted.numpy())
print(f"\nTest Accuracy: {accuracy*100:.2f}%")




Test Accuracy: 82.41%


In [26]:
torch.save(model.state_dict(), "sentiment_model.pth")
print("sentiment_model.pth")

sentiment_model.pth


In [27]:
def predict_sentiment(review, model):
    model.eval()
    cleaned_review = clean_text(review)
    vector = sentence_vector(cleaned_review)
    vector = torch.tensor(vector, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        prediction = model(vector).item()

    sentiment = "Positive " if prediction >= 0.5 else "Negative "

    print("\nReview:")
    print(review)
    print(f"\nPrediction: {sentiment}")
    print(f"Confidence: {prediction:.4f}")

predict_sentiment("This movie was amazing and very interesting", model)
predict_sentiment("This movie was boring and bad", model)


Review:
This movie was amazing and very interesting

Prediction: Positive 
Confidence: 0.9978

Review:
This movie was boring and bad

Prediction: Negative 
Confidence: 0.0018


In [28]:
torch.save(model.state_dict(), "sentiment_model.pth")

loaded_model = SentimentClassifier()
loaded_model.load_state_dict(torch.load("sentiment_model.pth"))
loaded_model.eval()

predict_sentiment("This movie was fantastic!", loaded_model)

from google.colab import files
files.download("sentiment_model.pth")


Review:
This movie was fantastic!

Prediction: Positive 
Confidence: 0.9825


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>